# Basic data cleaning (Information Systems)

Rule-based: clean author names, fill missing values, similarity-based standardization of addresses. No API.

In [ ]:
import pandas as pd
import re
from pathlib import Path

# Load Information Systems article data (same folder as this notebook)
CSV_PATH = Path("Information_Systems_article_data.csv")
df = pd.read_csv(CSV_PATH)

# Convert to lowercase and remove special characters for authors
def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df["Standardized_name"] = df["Author_name"].apply(clean_text)
print(df.head())

In [ ]:
df = df.fillna('NULL')

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Convert addresses to TF-IDF vectors (character n-grams) to find similar ones
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 4))
tfidf_matrix = vectorizer.fit_transform(df["Author_Address"].fillna(''))
print(tfidf_matrix.shape)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(tfidf_matrix)
similarity_df = pd.DataFrame(similarity_matrix, index=df["Author_Address"].fillna(''), columns=df["Author_Address"].fillna(''))
print(similarity_df.iloc[:5, :5])

In [ ]:
threshold = 0.75
matches = {}
addr = df["Author_Address"].fillna('')
for i in range(len(addr)):
    for j in range(len(addr)):
        if i != j and similarity_matrix[i][j] > threshold:
            key = addr.iloc[i]
            matches.setdefault(key, []).append(addr.iloc[j])
matches_df = pd.DataFrame(list(matches.items()), columns=["Original Address", "Similar Address"])
print(matches_df.head(10))

In [ ]:
def standardize_name(name, matches):
    if name in matches and isinstance(matches[name], list):
        valid = [str(n) for n in matches[name] if pd.notna(n)]
        if valid:
            return min(valid, key=len)
    return name

df["Standardized_Address"] = df["Author_Address"].fillna('').apply(lambda x: standardize_name(x, matches))
df.to_csv("Information_Systems_basic_cleaned.csv", index=False)
print(df[["Author_Address", "Standardized_Address"]].head())